In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

In [ ]:
print("Pytorch version",torch.__version__)
print("Gpu available",torch.cuda.is_available())

In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from datasets import load_dataset

# Loading the tokenizer and the model and the dataset

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2) # 2 labels: positive and negative

In [ ]:
print(model) # By default all are trainable

In [ ]:
imdb_data = load_dataset('imdb')
print(imdb_data)

In [ ]:
train_data = imdb_data['train']
test_data = imdb_data['test']

# Partition the train data into val and test splits

In [ ]:
from datasets import DatasetDict
# Split the train_data into train and validation sets 
# stratified by the label to ensure equal distribution of labels in both sets
train_val_split = train_data.train_test_split(test_size=0.1, seed=42, stratify_by_column='label')
train_data = train_val_split['train']
val_data = train_val_split['test']

In [ ]:
print("Train data size",len(train_data))
print("Validation data size",len(val_data))
print("Test data size",len(test_data))

# Cleaning (To be done)

# Tokenization

In [ ]:
def tokenize_function(examples):
    '''
    Tokenize the text column of the input examples
    Cuts the text if it is longer than the maximum length of the model
    The responsibilty of the padding is left to the dataloader
    '''
    return tokenizer(examples['text'], truncation=True)

# Making DataLoaders: We will also need to handle padding of different sizes

In [ ]:
batch_size = 8
lr=5e-5
num_epochs = 3

In [ ]:
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

In [ ]:
# Tokenize the datasets and remove the text column
tokenized_train= train_data.map(tokenize_function, batched=True)
tokenized_val= val_data.map(tokenize_function, batched=True)
tokenized_test= test_data.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.remove_columns(['text'])
tokenized_val = tokenized_val.remove_columns(['text'])
tokenized_test = tokenized_test.remove_columns(['text'])


In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

In [ ]:
train_dataloader = DataLoader(
    tokenized_train, batch_size=batch_size, shuffle=True, collate_fn=data_collator
)
val_dataloader = DataLoader(
    tokenized_val, batch_size=batch_size, collate_fn=data_collator
)
test_dataloader = DataLoader(
    tokenized_test, batch_size=batch_size, collate_fn=data_collator
)

# Model Training

In [ ]:
model = model.to('cuda')

In [ ]:
from transformers import TrainingArguments, Trainer

In [ ]:
training_args=TrainingArguments(
    output_dir=f'./results/batch_size={batch_size}_lr={lr}_epochs={num_epochs}/',          # output directory
    num_train_epochs=3,              # total number of training epochs
    per_device_train_batch_size=batch_size,  # batch size per device during training
    per_device_eval_batch_size=batch_size,   # batch size for evaluation
    learning_rate=5e-5,               # learning rate
    weight_decay=0.01,               # strength of weight decay
    logging_dir=f'./logs/batch_size={batch_size}_lr={lr}_epochs={num_epochs}/',            # directory for storing logs
    logging_steps=100,
    eval_strategy="epoch", # evaluate at the end of each epoch
    save_strategy="epoch", # save at the end of each epoch
    save_total_limit=3,
    load_best_model_at_end=True, # load the best model when finished training on the basis of evaluation metric
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="tensorboard",
)

In [ ]:
import evaluate  # Hugging Face evaluation module
accuracy = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()